# Notebook 01 - Detecção de Componentes

**Tech Challenge - Fase 5 - Hackaton FIAP**
**Modelagem de Ameaças utilizando IA (STRIDE)**

Este notebook executa os Passos 1 e 2 do pipeline:
- **Passo 1**: Identificar provedor cloud (AWS ou Azure) via Claude Vision
- **Passo 2a**: Detectar componentes visuais com Florence-2 (bounding boxes)
- **Passo 2b**: Classificar cada componente com Claude Vision (nome exato do serviço)

**GPU necessária**: T4 ou A100

## 1. Setup e Dependências

In [ ]:
# Instalar dependências
!pip install -q transformers anthropic supervision fpdf2 timm einops flash_attn

In [ ]:
import os
import json
import base64
import time
from pathlib import Path

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import supervision as sv
from transformers import AutoProcessor, AutoModelForCausalLM
import anthropic

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Montar Google Drive e Configurar Caminhos

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

# Caminhos no Google Drive
DRIVE_BASE = Path('/content/drive/MyDrive/hackaton-stride')
IMAGES_DIR = DRIVE_BASE / 'test_images'
OUTPUT_DIR = DRIVE_BASE / 'outputs' / 'detections'

# Criar diretórios se não existirem
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Chave API do Claude (configurar em Secrets do Colab)
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print(f"Diretório de imagens: {IMAGES_DIR}")
print(f"Diretório de output: {OUTPUT_DIR}")
print(f"Imagens disponíveis: {list(IMAGES_DIR.glob('*.png'))}")

In [ ]:
# Selecionar imagem de teste
# Altere o índice para testar com outra imagem (1 = AWS, 2 = Azure)
ARCH_INDEX = 1
IMAGE_PATH = IMAGES_DIR / f'arquitetura {ARCH_INDEX}.png'

# Carregar e exibir a imagem
image = Image.open(IMAGE_PATH).convert('RGB')
print(f"Imagem: {IMAGE_PATH.name}")
print(f"Tamanho: {image.size}")

plt.figure(figsize=(14, 10))
plt.imshow(image)
plt.title(f'Arquitetura {ARCH_INDEX}')
plt.axis('off')
plt.show()

## 3. Passo 1 — Identificar Provedor Cloud (Claude Vision)

In [ ]:
def encode_image_base64(image_path: str) -> str:
    """Codifica imagem em base64 para enviar ao Claude Vision."""
    with open(image_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def encode_pil_image_base64(pil_image: Image.Image) -> str:
    """Codifica imagem PIL em base64."""
    import io
    buffer = io.BytesIO()
    pil_image.save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

def identify_cloud_provider(image_path: str) -> str:
    """
    Passo 1: Identifica o provedor cloud (AWS ou Azure) a partir da imagem.
    Usa Claude Vision para analisar logotipos, cores e nomenclatura.
    """
    image_b64 = encode_image_base64(image_path)

    response = client.messages.create(
        model='claude-sonnet-4-6-20250514',
        max_tokens=100,
        messages=[
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'image',
                        'source': {
                            'type': 'base64',
                            'media_type': 'image/png',
                            'data': image_b64
                        }
                    },
                    {
                        'type': 'text',
                        'text': (
                            'Analise esta imagem de diagrama de arquitetura de software. '
                            'Identifique o provedor cloud principal baseando-se nos logotipos, '
                            'ícones, cores e nomes de serviços visíveis. '
                            'Responda APENAS com uma palavra: "AWS" ou "Azure".'
                        )
                    }
                ]
            }
        ]
    )

    provider = response.content[0].text.strip().upper()
    # Normalizar resposta
    if 'AWS' in provider:
        return 'AWS'
    elif 'AZURE' in provider:
        return 'Azure'
    else:
        print(f"Resposta inesperada: {provider}. Assumindo AWS.")
        return 'AWS'

In [ ]:
# Executar Passo 1
t0 = time.time()
provider = identify_cloud_provider(str(IMAGE_PATH))
t_provider = time.time() - t0

print(f"Provedor identificado: {provider}")
print(f"Tempo: {t_provider:.2f}s")

## 4. Passo 2a — Detecção de Componentes com Florence-2

In [ ]:
# Carregar modelo Florence-2
FLORENCE_MODEL_ID = 'microsoft/Florence-2-large'

print("Carregando Florence-2...")
florence_processor = AutoProcessor.from_pretrained(
    FLORENCE_MODEL_ID, trust_remote_code=True
)
florence_model = AutoModelForCausalLM.from_pretrained(
    FLORENCE_MODEL_ID, trust_remote_code=True,
    torch_dtype=torch.float16
).to('cuda')
print("Florence-2 carregado com sucesso!")

In [ ]:
def detect_components_florence(image: Image.Image, task: str = '<OD>') -> dict:
    """
    Passo 2a: Usa Florence-2 para detectar regiões de interesse na imagem.
    Retorna bounding boxes dos componentes detectados.

    Tasks disponíveis:
    - '<OD>': Object Detection (detecção geral)
    - '<DENSE_REGION_CAPTION>': Detecção + legenda densa
    - '<REGION_PROPOSAL>': Propostas de regiões
    """
    # Tentar múltiplas estratégias de detecção
    all_boxes = []
    all_labels = []

    tasks = ['<OD>', '<DENSE_REGION_CAPTION>', '<REGION_PROPOSAL>']

    for task_prompt in tasks:
        inputs = florence_processor(
            text=task_prompt,
            images=image,
            return_tensors='pt'
        ).to('cuda', torch.float16)

        with torch.no_grad():
            generated_ids = florence_model.generate(
                input_ids=inputs['input_ids'],
                pixel_values=inputs['pixel_values'],
                max_new_tokens=2048,
                num_beams=3
            )

        generated_text = florence_processor.batch_decode(
            generated_ids, skip_special_tokens=False
        )[0]

        result = florence_processor.post_process_generation(
            generated_text,
            task=task_prompt,
            image_size=(image.width, image.height)
        )

        # Extrair bboxes dependendo da task
        if task_prompt == '<OD>' and '<OD>' in result:
            data = result['<OD>']
            all_boxes.extend(data.get('bboxes', []))
            all_labels.extend(data.get('labels', []))
        elif task_prompt == '<DENSE_REGION_CAPTION>' and '<DENSE_REGION_CAPTION>' in result:
            data = result['<DENSE_REGION_CAPTION>']
            all_boxes.extend(data.get('bboxes', []))
            all_labels.extend(data.get('labels', []))
        elif task_prompt == '<REGION_PROPOSAL>' and '<REGION_PROPOSAL>' in result:
            data = result['<REGION_PROPOSAL>']
            all_boxes.extend(data.get('bboxes', []))
            all_labels.extend(data.get('labels', ['region'] * len(data.get('bboxes', []))))

    return {'bboxes': all_boxes, 'labels': all_labels}


def filter_and_merge_boxes(detections: dict, image_size: tuple,
                           min_area_ratio: float = 0.001,
                           max_area_ratio: float = 0.5,
                           iou_threshold: float = 0.5) -> dict:
    """
    Filtra bounding boxes por tamanho e remove duplicatas via NMS.
    """
    img_w, img_h = image_size
    img_area = img_w * img_h

    boxes = []
    labels = []

    for bbox, label in zip(detections['bboxes'], detections['labels']):
        x1, y1, x2, y2 = bbox
        w = x2 - x1
        h = y2 - y1
        area = w * h
        ratio = area / img_area

        # Filtrar por tamanho (nem muito pequeno, nem muito grande)
        if min_area_ratio <= ratio <= max_area_ratio and w > 10 and h > 10:
            boxes.append([x1, y1, x2, y2])
            labels.append(label)

    if not boxes:
        return {'bboxes': [], 'labels': []}

    # NMS simples para remover duplicatas
    boxes_np = np.array(boxes)
    keep = []
    used = set()

    for i in range(len(boxes_np)):
        if i in used:
            continue
        keep.append(i)
        for j in range(i + 1, len(boxes_np)):
            if j in used:
                continue
            # Calcular IoU
            xi1 = max(boxes_np[i][0], boxes_np[j][0])
            yi1 = max(boxes_np[i][1], boxes_np[j][1])
            xi2 = min(boxes_np[i][2], boxes_np[j][2])
            yi2 = min(boxes_np[i][3], boxes_np[j][3])
            inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
            area_i = (boxes_np[i][2] - boxes_np[i][0]) * (boxes_np[i][3] - boxes_np[i][1])
            area_j = (boxes_np[j][2] - boxes_np[j][0]) * (boxes_np[j][3] - boxes_np[j][1])
            iou = inter / (area_i + area_j - inter + 1e-6)
            if iou > iou_threshold:
                used.add(j)

    filtered_boxes = [boxes[i] for i in keep]
    filtered_labels = [labels[i] for i in keep]

    return {'bboxes': filtered_boxes, 'labels': filtered_labels}

In [ ]:
# Executar Passo 2a: Detecção com Florence-2
t0 = time.time()
raw_detections = detect_components_florence(image)
t_florence = time.time() - t0

print(f"Detecções brutas: {len(raw_detections['bboxes'])} regiões")
print(f"Tempo Florence-2: {t_florence:.2f}s")

# Filtrar e remover duplicatas
detections = filter_and_merge_boxes(raw_detections, image.size)
print(f"Detecções após filtro: {len(detections['bboxes'])} regiões")

In [ ]:
# Visualizar detecções brutas do Florence-2
fig, ax = plt.subplots(1, 1, figsize=(16, 12))
ax.imshow(image)

colors = plt.cm.Set3(np.linspace(0, 1, len(detections['bboxes'])))

for idx, (bbox, label) in enumerate(zip(detections['bboxes'], detections['labels'])):
    x1, y1, x2, y2 = bbox
    w = x2 - x1
    h = y2 - y1
    rect = patches.Rectangle(
        (x1, y1), w, h,
        linewidth=2, edgecolor=colors[idx], facecolor='none'
    )
    ax.add_patch(rect)
    ax.text(x1, y1 - 5, f'{idx}: {label}', fontsize=7,
            color='white', backgroundcolor=colors[idx])

ax.set_title(f'Florence-2: {len(detections["bboxes"])} regiões detectadas')
ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Passo 2b — Classificação de Componentes com Claude Vision

In [ ]:
# Classes de detecção do projeto
COMPONENT_CLASSES = {
    'waf_firewall': ['AWS WAF', 'AWS Shield', 'Azure Firewall'],
    'cdn': ['Amazon CloudFront', 'Azure CDN'],
    'load_balancer': ['ALB', 'NLB', 'Azure Load Balancer'],
    'vpc_vnet': ['VPC', 'VNet', 'Subnet'],
    'compute': ['EC2', 'SEI/SIP', 'App Service', 'VM'],
    'auto_scaling': ['Auto Scaling Group', 'VMSS'],
    'orchestrator': ['Logic Apps', 'Step Functions', 'Lambda'],
    'database': ['RDS', 'Aurora', 'Azure SQL', 'Cosmos DB'],
    'cache': ['ElastiCache', 'Azure Cache for Redis'],
    'storage': ['S3', 'EFS', 'Azure Blob', 'NFS'],
    'api_gateway': ['API Gateway', 'Azure API Management'],
    'developer_portal': ['Developer Portal'],
    'web_service': ['REST', 'SOAP', 'SaaS Service'],
    'auth_identity': ['IAM', 'Microsoft Entra', 'Cognito'],
    'kms_encryption': ['AWS KMS', 'Azure Key Vault'],
    'monitoring': ['CloudWatch', 'CloudTrail', 'Azure Monitor'],
    'backup': ['AWS Backup', 'Azure Backup'],
    'messaging': ['SES', 'SQS', 'SNS', 'Azure Service Bus'],
    'user_actor': ['Usuário', 'Developer', 'Client'],
}

# Lista plana de todos os serviços para o prompt
ALL_SERVICES = []
for cls, services in COMPONENT_CLASSES.items():
    for svc in services:
        ALL_SERVICES.append(f'{svc} ({cls})')

SERVICES_LIST = ', '.join(ALL_SERVICES)

In [ ]:
def classify_component(crop_image: Image.Image, provider: str,
                       full_image_b64: str) -> dict:
    """
    Passo 2b: Classifica um crop de componente usando Claude Vision.
    Envia o crop + imagem completa para contexto.

    Retorna dict com 'name' (nome do serviço) e 'class' (classe genérica).
    """
    crop_b64 = encode_pil_image_base64(crop_image)

    prompt = f"""Analise este recorte de um diagrama de arquitetura {provider}.

A segunda imagem mostra o diagrama completo para contexto.

Identifique o componente/serviço cloud mostrado no recorte.

Serviços possíveis: {SERVICES_LIST}

Responda APENAS em JSON válido com este formato:
{{
  "name": "Nome exato do serviço (ex: Amazon RDS, AWS WAF)",
  "class": "classe genérica (ex: database, waf_firewall)"
}}

Se o recorte não contiver um componente cloud identificável, responda:
{{"name": "unknown", "class": "unknown"}}"""

    response = client.messages.create(
        model='claude-sonnet-4-6-20250514',
        max_tokens=200,
        messages=[
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'image',
                        'source': {
                            'type': 'base64',
                            'media_type': 'image/png',
                            'data': crop_b64
                        }
                    },
                    {
                        'type': 'image',
                        'source': {
                            'type': 'base64',
                            'media_type': 'image/png',
                            'data': full_image_b64
                        }
                    },
                    {
                        'type': 'text',
                        'text': prompt
                    }
                ]
            }
        ]
    )

    text = response.content[0].text.strip()
    # Extrair JSON da resposta
    try:
        # Tentar encontrar JSON na resposta
        start = text.find('{')
        end = text.rfind('}') + 1
        if start >= 0 and end > start:
            return json.loads(text[start:end])
    except json.JSONDecodeError:
        pass

    return {'name': 'unknown', 'class': 'unknown'}

In [ ]:
# Executar Passo 2b: Classificar cada componente
print(f"Classificando {len(detections['bboxes'])} componentes com Claude Vision...")
print(f"Provedor: {provider}\n")

full_image_b64 = encode_image_base64(str(IMAGE_PATH))
components = []

t0 = time.time()
for idx, bbox in enumerate(detections['bboxes']):
    x1, y1, x2, y2 = [int(c) for c in bbox]

    # Recortar região com margem
    margin = 10
    crop_x1 = max(0, x1 - margin)
    crop_y1 = max(0, y1 - margin)
    crop_x2 = min(image.width, x2 + margin)
    crop_y2 = min(image.height, y2 + margin)
    crop = image.crop((crop_x1, crop_y1, crop_x2, crop_y2))

    # Classificar com Claude Vision
    result = classify_component(crop, provider, full_image_b64)

    component = {
        'id': idx,
        'name': result.get('name', 'unknown'),
        'class': result.get('class', 'unknown'),
        'bbox': [x1, y1, x2 - x1, y2 - y1],  # [x, y, w, h]
        'provider': provider,
        'florence_label': detections['labels'][idx]
    }
    components.append(component)

    status = f"  [{idx+1}/{len(detections['bboxes'])}] {result.get('name', '?')} ({result.get('class', '?')})"
    print(status)

    # Rate limiting
    time.sleep(0.5)

t_classify = time.time() - t0

# Filtrar componentes desconhecidos
known_components = [c for c in components if c['class'] != 'unknown']
print(f"\nComponentes identificados: {len(known_components)}/{len(components)}")
print(f"Tempo classificação: {t_classify:.2f}s")

## 6. Salvar Resultados

In [ ]:
# Montar JSON de saída
output_data = {
    'image': IMAGE_PATH.name,
    'provider': provider,
    'image_size': {'width': image.width, 'height': image.height},
    'total_detections': len(detections['bboxes']),
    'components': known_components,
    'timing': {
        'provider_identification_s': round(t_provider, 2),
        'florence_detection_s': round(t_florence, 2),
        'claude_classification_s': round(t_classify, 2),
        'total_s': round(t_provider + t_florence + t_classify, 2)
    }
}

# Salvar JSON
output_file = OUTPUT_DIR / f'componentes_arquitetura_{ARCH_INDEX}.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"Resultados salvos em: {output_file}")
print(f"\nResumo:")
print(f"  Provedor: {provider}")
print(f"  Componentes detectados: {len(known_components)}")
print(f"  Tempo total: {output_data['timing']['total_s']}s")
print(f"\nComponentes:")
for c in known_components:
    print(f"  - {c['name']} ({c['class']})")

## 7. Visualização com Supervision

In [ ]:
# Visualizar componentes detectados com bounding boxes
if known_components:
    # Preparar dados para supervision
    xyxy = np.array([
        [c['bbox'][0], c['bbox'][1],
         c['bbox'][0] + c['bbox'][2], c['bbox'][1] + c['bbox'][3]]
        for c in known_components
    ])

    detections_sv = sv.Detections(
        xyxy=xyxy,
        class_id=np.arange(len(known_components))
    )

    labels = [
        f"{c['name']}\n({c['class']})"
        for c in known_components
    ]

    # Anotar imagem
    img_np = np.array(image)

    box_annotator = sv.BoxAnnotator(
        thickness=2
    )
    label_annotator = sv.LabelAnnotator(
        text_scale=0.4,
        text_padding=4
    )

    annotated = box_annotator.annotate(
        scene=img_np.copy(),
        detections=detections_sv
    )
    annotated = label_annotator.annotate(
        scene=annotated,
        detections=detections_sv,
        labels=labels
    )

    # Exibir
    plt.figure(figsize=(18, 14))
    plt.imshow(annotated)
    plt.title(f'Arquitetura {ARCH_INDEX} ({provider}) - {len(known_components)} componentes detectados')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # Salvar imagem anotada
    annotated_path = OUTPUT_DIR / f'annotated_arquitetura_{ARCH_INDEX}.png'
    Image.fromarray(annotated).save(annotated_path)
    print(f"Imagem anotada salva em: {annotated_path}")
else:
    print("Nenhum componente identificado para visualizar.")

## 8. Resumo da Execução

In [ ]:
# Tabela resumo
print("=" * 60)
print(f"RESUMO - Detecção de Componentes")
print("=" * 60)
print(f"Imagem: {IMAGE_PATH.name}")
print(f"Provedor: {provider}")
print(f"Tamanho: {image.size}")
print(f"")
print(f"Detecções Florence-2: {len(detections['bboxes'])}")
print(f"Componentes classificados: {len(known_components)}")
print(f"")
print(f"Tempo - Identificação provedor: {t_provider:.2f}s")
print(f"Tempo - Florence-2: {t_florence:.2f}s")
print(f"Tempo - Classificação Claude: {t_classify:.2f}s")
print(f"Tempo total: {t_provider + t_florence + t_classify:.2f}s")
print(f"")
print(f"Output: {output_file}")
print("=" * 60)

# Classes detectadas
from collections import Counter
class_counts = Counter(c['class'] for c in known_components)
print(f"\nClasses detectadas:")
for cls, count in class_counts.most_common():
    print(f"  {cls}: {count}")